In [0]:
# Read Bronze Delta table

bronze_df = spark.table("walmart_retail.sales.bronze_sales")

print("Bronze rows:", bronze_df.count())
print("Bronze columns:", len(bronze_df.columns))

display(bronze_df.limit(10))

Bronze rows: 8399
Bronze columns: 25


city,customer_age,customer_name,customer_segment,discount,number_of_records,order_date,order_id,order_priority,order_quantity,product_base_margin,product_category,product_container,product_name,product_sub_category,profit,region,row_id,sales,ship_date,ship_mode,shipping_cost,state,unit_price,zip_code
Granite City,60.0,Roy Skaria,Corporate,0.05,1,2015-01-02T00:00:00.000Z,29030,Medium,10,0.44,Office Supplies,Wrap Bag,Prismacolor Color Pencil Set,Pens & Art Supplies,24.6,Central,4075,192.58,2015-01-03T00:00:00.000Z,Regular Air,4.1,Illinois,19.84,62040
Sierra Vista,60.0,Dario Medina,Small Business,0.1,1,2015-01-02T00:00:00.000Z,47591,Critical,27,0.58,Office Supplies,Small Box,Sterilite Officeware Hinged File Box,Storage & Organization,-92.25,West,6690,279.4,2015-01-03T00:00:00.000Z,Regular Air,6.91,Arizona,10.48,85635
Caldwell,60.0,Resi Polking,Small Business,0.04,1,2015-01-02T00:00:00.000Z,54214,Medium,38,0.46,Technology,Large Box,Canon PC-428 Personal Copier,Copiers and Fax,1899.23,West,7580,7325.63,2015-01-05T00:00:00.000Z,Express Air,24.49,Idaho,199.99,83605
Pahrump,60.0,Chuck Clark,Corporate,0.1,1,2015-01-03T00:00:00.000Z,35811,Critical,49,0.38,Office Supplies,Small Box,Park Ridge™ Embossed Executive Business Envelopes,Envelopes,352.41,West,5017,741.57,2015-01-04T00:00:00.000Z,Regular Air,1.39,Nevada,15.57,89041
Laguna Hills,60.0,Jill Matthias,Consumer,0.09,1,2015-01-03T00:00:00.000Z,43494,Medium,2,0.38,Technology,Medium Box,Polycom ViewStation™ Adapter H323 Videoconferencing Unit,Office Machines,-4858.6716,West,6146,3668.28,2015-01-04T00:00:00.000Z,Regular Air,13.99,California,1938.02,92653
Revere,60.0,Deborah Brumfield,Corporate,0.02,1,2015-01-03T00:00:00.000Z,50471,Low,25,0.59,Furniture,Jumbo Drum,Hon GuestStacker Chair,Chairs & Chairmats,1358.53,East,7074,5859.25,2015-01-05T00:00:00.000Z,Delivery Truck,28.16,MA,226.67,2151
Pahrump,60.0,Chuck Clark,Corporate,0.1,1,2015-01-03T00:00:00.000Z,35811,Critical,6,0.56,Office Supplies,Wrap Bag,Newell 310,Pens & Art Supplies,-3.24,West,5018,10.39,2015-01-05T00:00:00.000Z,Regular Air,0.7,Nevada,1.76,89041
Lansing,60.0,Ralph Knight,Consumer,0.04,1,2015-01-03T00:00:00.000Z,10945,Medium,14,0.57,Technology,Small Box,600 Series Flip,Telephones and Communication,4.221,Central,1513,1170.025,2015-01-05T00:00:00.000Z,Regular Air,8.99,Illinois,95.99,60438
Danville,60.0,Eva Jacobs,Home Office,0.03,1,2015-01-03T00:00:00.000Z,16164,Low,26,0.36,Technology,Jumbo Drum,Lexmark Z54se Color Inkjet Printer,Office Machines,774.79,South,2241,2354.15,2015-01-07T00:00:00.000Z,Delivery Truck,14.0,Kentucky,90.97,40422
Broomfield,60.0,Eva Jacobs,Home Office,0.08,1,2015-01-03T00:00:00.000Z,16164,Low,26,0.44,Office Supplies,Wrap Bag,"Binney & Smith inkTank™ Erasable Desk Highlighter, Chisel Tip, Yellow, 12/Box",Pens & Art Supplies,-70.73,West,2240,69.38,2015-01-10T00:00:00.000Z,Regular Air,4.28,Colorado,2.52,80020


In [0]:
# Inspect Bronze schema

bronze_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- customer_age: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- number_of_records: long (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_priority: string (nullable = true)
 |-- order_quantity: long (nullable = true)
 |-- product_base_margin: double (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_container: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_sub_category: string (nullable = true)
 |-- profit: double (nullable = true)
 |-- region: string (nullable = true)
 |-- row_id: long (nullable = true)
 |-- sales: double (nullable = true)
 |-- ship_date: timestamp (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- state: string (nullable = true)
 |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType, StringType, DoubleType

silver_df = (
    bronze_df
    # Dates
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("ship_date", F.to_date("ship_date"))

    # Numeric types
    .withColumn("customer_age", F.col("customer_age").cast("int"))
    .withColumn("order_id", F.col("order_id").cast("long"))
    .withColumn("order_quantity", F.col("order_quantity").cast("long"))
    .withColumn("row_id", F.col("row_id").cast("long"))
    .withColumn("zip_code", F.col("zip_code").cast("string"))

    # Clean text fields
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("customer_segment", F.trim(F.col("customer_segment")))
    .withColumn("order_priority", F.trim(F.col("order_priority")))
    .withColumn("product_category", F.trim(F.col("product_category")))
    .withColumn("product_container", F.trim(F.col("product_container")))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("product_sub_category", F.trim(F.col("product_sub_category")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("ship_mode", F.trim(F.col("ship_mode")))
    .withColumn("state", F.trim(F.col("state")))
)

print("Silver transformation applied.")
display(silver_df.limit(10))

Silver transformation applied.


city,customer_age,customer_name,customer_segment,discount,number_of_records,order_date,order_id,order_priority,order_quantity,product_base_margin,product_category,product_container,product_name,product_sub_category,profit,region,row_id,sales,ship_date,ship_mode,shipping_cost,state,unit_price,zip_code
Granite City,60,Roy Skaria,Corporate,0.05,1,2015-01-02,29030,Medium,10,0.44,Office Supplies,Wrap Bag,Prismacolor Color Pencil Set,Pens & Art Supplies,24.6,Central,4075,192.58,2015-01-03,Regular Air,4.1,Illinois,19.84,62040
Sierra Vista,60,Dario Medina,Small Business,0.1,1,2015-01-02,47591,Critical,27,0.58,Office Supplies,Small Box,Sterilite Officeware Hinged File Box,Storage & Organization,-92.25,West,6690,279.4,2015-01-03,Regular Air,6.91,Arizona,10.48,85635
Caldwell,60,Resi Polking,Small Business,0.04,1,2015-01-02,54214,Medium,38,0.46,Technology,Large Box,Canon PC-428 Personal Copier,Copiers and Fax,1899.23,West,7580,7325.63,2015-01-05,Express Air,24.49,Idaho,199.99,83605
Pahrump,60,Chuck Clark,Corporate,0.1,1,2015-01-03,35811,Critical,49,0.38,Office Supplies,Small Box,Park Ridge™ Embossed Executive Business Envelopes,Envelopes,352.41,West,5017,741.57,2015-01-04,Regular Air,1.39,Nevada,15.57,89041
Laguna Hills,60,Jill Matthias,Consumer,0.09,1,2015-01-03,43494,Medium,2,0.38,Technology,Medium Box,Polycom ViewStation™ Adapter H323 Videoconferencing Unit,Office Machines,-4858.6716,West,6146,3668.28,2015-01-04,Regular Air,13.99,California,1938.02,92653
Revere,60,Deborah Brumfield,Corporate,0.02,1,2015-01-03,50471,Low,25,0.59,Furniture,Jumbo Drum,Hon GuestStacker Chair,Chairs & Chairmats,1358.53,East,7074,5859.25,2015-01-05,Delivery Truck,28.16,MA,226.67,2151
Pahrump,60,Chuck Clark,Corporate,0.1,1,2015-01-03,35811,Critical,6,0.56,Office Supplies,Wrap Bag,Newell 310,Pens & Art Supplies,-3.24,West,5018,10.39,2015-01-05,Regular Air,0.7,Nevada,1.76,89041
Lansing,60,Ralph Knight,Consumer,0.04,1,2015-01-03,10945,Medium,14,0.57,Technology,Small Box,600 Series Flip,Telephones and Communication,4.221,Central,1513,1170.025,2015-01-05,Regular Air,8.99,Illinois,95.99,60438
Danville,60,Eva Jacobs,Home Office,0.03,1,2015-01-03,16164,Low,26,0.36,Technology,Jumbo Drum,Lexmark Z54se Color Inkjet Printer,Office Machines,774.79,South,2241,2354.15,2015-01-07,Delivery Truck,14.0,Kentucky,90.97,40422
Broomfield,60,Eva Jacobs,Home Office,0.08,1,2015-01-03,16164,Low,26,0.44,Office Supplies,Wrap Bag,"Binney & Smith inkTank™ Erasable Desk Highlighter, Chisel Tip, Yellow, 12/Box",Pens & Art Supplies,-70.73,West,2240,69.38,2015-01-10,Regular Air,4.28,Colorado,2.52,80020


In [0]:
# Silver data quality checks

print("Total rows:", silver_df.count())

print("\nNull counts:")
null_counts = silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
])

display(null_counts)

print("\nDuplicate row count:")
duplicate_count = silver_df.count() - silver_df.dropDuplicates().count()
print(duplicate_count)

print("\nInvalid quantity records:")
invalid_quantity = silver_df.filter(
    F.col("order_quantity") <= 0
).count()
print(invalid_quantity)

print("\nInvalid sales records:")
invalid_sales = silver_df.filter(
    F.col("sales") < 0
).count()
print(invalid_sales)

print("\nInvalid discount records:")
invalid_discount = silver_df.filter(
    (F.col("discount") < 0) | (F.col("discount") > 1)
).count()
print(invalid_discount)

print("\nInvalid date records:")
invalid_dates = silver_df.filter(
    F.col("ship_date") < F.col("order_date")
).count()
print(invalid_dates)

Total rows: 8399

Null counts:


city,customer_age,customer_name,customer_segment,discount,number_of_records,order_date,order_id,order_priority,order_quantity,product_base_margin,product_category,product_container,product_name,product_sub_category,profit,region,row_id,sales,ship_date,ship_mode,shipping_cost,state,unit_price,zip_code
0,903,0,0,0,0,0,0,0,0,63,0,0,0,0,0,0,0,0,0,0,0,0,0,0



Duplicate row count:
0

Invalid quantity records:
0

Invalid sales records:
0

Invalid discount records:
0

Invalid date records:
0


In [0]:
silver_table = "walmart_retail.sales.silver_sales"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Silver table created: {silver_table}")

Silver table created: walmart_retail.sales.silver_sales


In [0]:
silver_check = spark.table("walmart_retail.sales.silver_sales")

print("Silver rows:", silver_check.count())
print("Silver columns:", len(silver_check.columns))

display(silver_check.limit(10))

Silver rows: 8399
Silver columns: 25


city,customer_age,customer_name,customer_segment,discount,number_of_records,order_date,order_id,order_priority,order_quantity,product_base_margin,product_category,product_container,product_name,product_sub_category,profit,region,row_id,sales,ship_date,ship_mode,shipping_cost,state,unit_price,zip_code
McKeesport,null,Jessica Myrick,Small Business,0.1,1,2012-01-01,28774,High,32,0.68,Office Supplies,Small Box,"Perma STOR-ALL™ Hanging File Box, 13 1/8""W x 12 1/4""D x 10 1/2""H",Storage & Organization,-111.8,East,4031,180.36,2012-01-02,Regular Air,4.69,Pennsylvania,5.98,15131
Bowie,null,Matt Collister,Home Office,0.08,1,2012-01-01,13729,Not Specified,9,null,Office Supplies,Large Box,Safco Industrial Wire Shelving,Storage & Organization,-342.91,East,1914,872.48,2012-01-03,Express Air,35.0,Maryland,95.99,20715
Napa,null,Alan Schoenberger,Corporate,0.0,1,2012-01-02,37537,Low,4,0.56,Furniture,Jumbo Drum,Hon 4070 Series Pagoda™ Armless Upholstered Stacking Chairs,Chairs & Chairmats,-193.08,West,5272,1239.06,2012-01-02,Delivery Truck,48.8,California,291.73,94559
Montebello,null,Elizabeth Moffitt,Consumer,0.08,1,2012-01-02,44069,Critical,43,0.39,Office Supplies,Wrap Bag,White GlueTop Scratch Pads,Paper,247.79,West,6225,614.8,2012-01-02,Regular Air,1.97,California,15.04,90640
Napa,null,Alan Schoenberger,Corporate,0.07,1,2012-01-02,37537,Low,43,0.69,Furniture,Jumbo Drum,Hon Valutask™ Swivel Chairs,Chairs & Chairmats,-1049.85,West,5273,4083.19,2012-01-04,Delivery Truck,45.0,California,100.98,94559
Montebello,null,Elizabeth Moffitt,Consumer,0.09,1,2012-01-02,44069,Critical,16,0.4,Office Supplies,Wrap Bag,"Black Print Carbonless Snap-Off® Rapid Letter, 8 1/2"" x 7""",Paper,26.71,West,6224,137.63,2012-01-04,Express Air,2.15,California,9.11,90640
Prior Lake,null,David Philippe,Consumer,0.06,1,2012-01-02,9285,Critical,3,0.36,Office Supplies,Small Box,"Avery Trapezoid Ring Binder, 3"" Capacity, Black, 1040 sheets",Binders and Binder Accessories,-11.937,Central,1279,124.81,2012-01-04,Regular Air,2.99,Minnesota,40.98,55372
Napa,null,Alan Schoenberger,Corporate,0.05,1,2012-01-02,37537,Low,32,0.59,Office Supplies,Small Box,"Dual Level, Single-Width Filing Carts",Storage & Organization,1438.49,West,5274,4902.38,2012-01-09,Regular Air,7.07,California,155.06,94559
Phenix City,null,Patrick Jones,Home Office,0.09,1,2012-01-03,40354,High,4,0.64,Furniture,Jumbo Box,Bush Advantage Collection® Round Conference Table,Tables,-93.16,South,5705,698.0,2012-01-04,Delivery Truck,52.2,Alabama,212.6,36869
Draper,null,Larry Tron,Home Office,0.05,1,2012-01-03,9762,High,12,0.78,Furniture,Medium Box,36X48 HARDFLOOR CHAIRMAT,Office Furnishings,-146.05,West,1336,262.76,2012-01-04,Regular Air,21.2,Utah,20.98,84020
